In [0]:
pip install langchain langchain-community langchain-huggingface chromadb sentence-transformers

In [0]:
print('test')

In [0]:
pip install pypdf

In [0]:
# pip install langchain-community langchain-text-splitters pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Load PDF
pdf_files = ["/Workspace/Users/surbhiwahie@gmail.com/Rag-System---Ask-My-Doc/Data/raw/Diabetes Care Booklet.pdf"]
all_docs = []

for file in pdf_files:
    loader = PyPDFLoader(file)
    docs = loader.load()
    
    # Add source info
    for d in docs:
        d.metadata["source"] = file.split("/")[-1]
    
    all_docs.extend(docs)

print(f"Loaded {len(all_docs)} pages total")

In [0]:
# 2. Chunking

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = splitter.split_documents(all_docs)

print(f"Created {len(chunks)} chunks")

In [0]:
import shutil
shutil.rmtree("/tmp/chroma_db", ignore_errors=True)

In [0]:
# 3. Add metadata
for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i

# 4. Embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# 5. Store in Chroma
db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="/tmp/chroma_db"
)
db.persist()

print("✅ Vector DB created!")

In [0]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 1. Embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

# 2. Load DB
db = Chroma(
    persist_directory="/tmp/chroma_db",
    embedding_function=embedding_model
)

# 3. Retriever
retriever = db.as_retriever(search_kwargs={"k": 3})

# 4. Query
query = "symptoms of diabetes mellitus list"
results = retriever.invoke(query)

# 5. Print
for r in results:
    print("----")
    print(r.page_content[:300])
    print(r.metadata)

In [0]:
pip install openai langchain-openai

In [0]:
pip install python-dotenv